# Rate Comparison Analysis (Sanitised)

This notebook is a portfolio-safe version of an operational rate comparison workflow.  
Sensitive site identifiers, internal process names, and local file references have been replaced with generic placeholders while preserving the analytical approach.


In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
df = pd.read_csv("path/to/rate_comparison_data.csv")
df.head(20)

In [ ]:
df.dtypes

In [ ]:
df_needed = df[['shift', 'day', 'process_group', 'interval_start_local', 'interval_end_local', 'dow', 'week', 'year_number_sun', 'predicted_rate', 'actual_rate', 'planned_rate', 'variance_flag']]

In [ ]:
df_needed.head()

In [ ]:
df_needed['process_group'].value_counts()

In [ ]:
df_needed = df_needed[df_needed['process_group'].isin(['PROCESS_A', 'PROCESS_B', 'PROCESS_C', 'PROCESS_D'])]

In [ ]:
df_needed['day'] = df_needed['day'].str.rstrip("T00:00:00.000Z")

df_needed.head()

In [ ]:
df_needed['interval_start_local'] = df_needed['interval_start_local'].str.replace('T',' ')
df_needed['interval_start_local'] = df_needed['interval_start_local'].str.replace('Z','')
df_needed['interval_end_local'] = df_needed['interval_end_local'].str.replace('T',' ')
df_needed['interval_end_local'] = df_needed['interval_end_local'].str.replace('Z','')


df_needed.head(10)


In [ ]:
df_needed['interval_end_local'] = pd.to_datetime(df_needed['interval_end_local'])
df_needed['interval_start_local'] = pd.to_datetime(df_needed['interval_start_local'])
df_needed['day'] = pd.to_datetime(df_needed['day'])

df_needed.dtypes

In [ ]:
df_needed = df_needed.sort_values(by = ['shift','interval_start_local','interval_end_local', 'dow'])

In [ ]:
df_needed['day'] = df_needed['interval_start_local'].dt.date
df_needed['interval_start_local'] = df_needed['interval_start_local'].dt.time
df_needed['interval_end_local'] = df_needed['interval_end_local'].dt.time

df_needed.head(10)



In [ ]:
df_needed = df_needed.drop(columns="year_number_sun")

In [ ]:
df_needed['shift'] = df_needed['shift'].str.strip()

In [ ]:
df_needed['shift'].str.len()

In [ ]:
df_ds = df_needed[df_needed['shift'] == 'Day']
df_ns = df_needed[df_needed['shift'] == 'Night']

In [ ]:
df_ds

In [ ]:
df_ns

In [ ]:
def assign_shift_type(dow):
    if hasattr(dow, 'strftime'):
        dow = dow.strftime('%a')
    day = dow.strip().title()

    front_end = ['Sun', 'Mon', 'Tue']
    back_end = ['Thu', 'Fri', 'Sat']

    if day in front_end:
        return 'Front End'
    elif day in back_end:
        return 'Back End'
    elif day == 'Wed':
        return 'Midweek'
    else:
        return 'unknown'

In [ ]:
df_ds['schedule_group'] = df_ds['dow'].apply(assign_shift_type)
df_ns['schedule_group'] = df_ns['dow'].apply(assign_shift_type)

In [ ]:
df_ns

In [ ]:
df_ds

In [ ]:
df_needed['schedule_group'] = df_needed['dow'].apply(assign_shift_type)

In [ ]:
df_needed.head(50)

In [ ]:
df_cleaned = df_needed[df_needed['planned_rate'] != 'null']

In [ ]:
cleaned_ds = df_ds[df_ds['planned_rate'] != 'null']
cleaned_ns = df_ns[df_ns['planned_rate'] != 'null']

In [ ]:
cleaned_ds

In [ ]:
cleaned_ns

In [ ]:
df_summary = df_cleaned.groupby(['process_group', 'schedule_group', 'shift', 'interval_start_local', 'interval_end_local']).agg({'predicted_rate':'mean', 'actual_rate': 'mean', 'planned_rate':'mean'}).reset_index().round(3)
df_summary.head(40)

In [ ]:
df_summary['Shift_View'] = (df_summary['schedule_group'] + ' ' + df_summary['shift'])
df_summary.head(10)

In [ ]:
df_summary = df_summary.drop(columns=['schedule_group', 'shift'])

In [ ]:
df_summary.head(150)

In [ ]:
df_summary['interval_start_local'] = df_summary['interval_start_local'].apply(
    lambda x: x.strftime('%H:%M GMT') if pd.notna(x) else None
)

In [ ]:
df_summary['interval_end_local'] = df_summary['interval_end_local'].apply(
    lambda x: x.strftime('%H:%M GMT') if pd.notna(x) else None
)

In [ ]:
df_summary

In [ ]:
process_a_summary = df_summary[df_summary['process_group'] == 'PROCESS_A']

In [ ]:
process_c_summary = df_summary[df_summary['process_group'] == 'PROCESS_C']
process_d_summary = df_summary[df_summary['process_group'] == 'PROCESS_D']
process_b_summary = df_summary[df_summary['process_group'] == 'PROCESS_B']

In [ ]:
process_a_summary = process_a_summary.sort_values(by='interval_start_local')

In [ ]:
process_a_time = process_a_summary.groupby('interval_start_local').agg({'predicted_rate':'mean','actual_rate': 'mean','planned_rate':'mean'}).reset_index().round(3)

In [ ]:
df_long = process_a_time.melt(
    id_vars='interval_start_local',
    value_vars=['predicted_rate', 'actual_rate', 'planned_rate'],
    var_name='rate_type',
    value_name='rate'
)

fig = px.line(
    df_long,
    x='interval_start_local',
    y='rate',
    color='rate_type',
    markers=True,
    text='rate',
    title='Average Rate Comparison by Time for Process A',
    labels={
        'interval_start_local': 'Time Interval',
        'rate': 'Average Rate',
        'rate_type': 'Rate Type'
    }
)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
process_b_time = process_b_summary.groupby('interval_start_local').agg({'predicted_rate':'mean','actual_rate': 'mean','planned_rate':'mean'}).reset_index().round(3)

In [ ]:
process_d_time = process_d_summary.groupby('interval_start_local').agg({'predicted_rate':'mean','actual_rate': 'mean','planned_rate':'mean'}).reset_index().round(3)
process_c_time = process_c_summary.groupby('interval_start_local').agg({'predicted_rate':'mean','actual_rate': 'mean','planned_rate':'mean'}).reset_index().round(3)

In [ ]:
process_d_long = process_d_time.melt(
    id_vars='interval_start_local',
    value_vars=['predicted_rate', 'actual_rate', 'planned_rate'],
    var_name='rate_type',
    value_name='rate'
)

fig = px.line(
    process_d_long,
    x='interval_start_local',
    y='rate',
    color='rate_type',
    markers=True,
    text='rate',
    title='Average Rate Comparison by Time for Process D',
    labels={
        'interval_start_local': 'Time Interval',
        'rate': 'Average Rate',
        'rate_type': 'Rate Type'
    }
)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
process_b_long = process_b_time.melt(
    id_vars='interval_start_local',
    value_vars=['predicted_rate', 'actual_rate', 'planned_rate'],
    var_name='rate_type',
    value_name='rate'
)

fig = px.line(
    process_b_long,
    x='interval_start_local',
    y='rate',
    color='rate_type',
    markers=True,
    text='rate',
    title='Average Rate Comparison by Time for Process B',
    labels={
        'interval_start_local': 'Time Interval',
        'rate': 'Average Rate',
        'rate_type': 'Rate Type'
    }
)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
process_c_long = process_c_time.melt(
    id_vars='interval_start_local',
    value_vars=['predicted_rate', 'actual_rate', 'planned_rate'],
    var_name='rate_type',
    value_name='rate'
)

fig = px.line(
    process_c_long,
    x='interval_start_local',
    y='rate',
    color='rate_type',
    markers=True,
    text='rate',
    title='Average Rate Comparison by Time for Process C',
    labels={
        'interval_start_local': 'Time Interval',
        'rate': 'Average Rate',
        'rate_type': 'Rate Type'
    }
)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
df_summary.head(60)

In [ ]:
agg_table = df_summary.groupby(['process_group', 'Shift_View']).agg({'predicted_rate':'mean', 'actual_rate':'mean', 'planned_rate':'mean'}).reset_index().round(3)

In [ ]:
agg_table['predicted_vs_actual_pct_dev'] = ((agg_table['predicted_rate'] - agg_table['actual_rate']) / agg_table['actual_rate']) * 100

In [ ]:
agg_table['predicted_vs_actual_pct_dev'] = agg_table['predicted_vs_actual_pct_dev'].map('{:.2f}%'.format)
agg_table.head(40)